# 01 — Verify raw tables

Checks that the bronze files loaded into `raw` look complete:
row counts per season for each raw table, compared with expected ranges.

In [10]:
import duckdb


con = duckdb.connect(DB_PATH, read_only=True)


In [11]:
df = con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""").df()

display(df)


,table_schema,table_name
0,raw,constructor_standings
1,raw,driver_standings
2,raw,race_winners


In [12]:
con.sql("""
    SELECT
        payload ->> '$.MRData.RaceTable.season'                   AS season,
        json_array_length(payload -> '$.MRData.RaceTable.Races') AS race_count
    FROM raw.race_winners
    ORDER BY season
""").df()


,season,race_count
0,2006,18
1,2007,17
2,2008,18
3,2009,17
4,2010,19
5,2011,19
6,2012,20
7,2013,19
8,2014,19
9,2015,19


In [13]:
# Drivers per season in driver_standings. Expected: 20 rows, at least 20 drivers each.
con.sql("""
    SELECT
        payload ->> '$.MRData.StandingsTable.season'                  AS season,
        json_array_length(payload -> '$.MRData.StandingsTable.StandingsLists[0].DriverStandings')       AS driver_count
    FROM raw.driver_standings
    ORDER BY season
""").df()


,season,driver_count
0,2006,27
1,2007,26
2,2008,22
3,2009,25
4,2010,27
5,2011,28
6,2012,25
7,2013,23
8,2014,24
9,2015,22


In [14]:
# Constructors per season in constructor_standings. Expected: 10 rows, at least 12 constructors each.
con.sql("""
    SELECT
        payload ->> '$.MRData.StandingsTable.season'  AS season,
        json_array_length(payload -> '$.MRData.StandingsTable.StandingsLists[0].ConstructorStandings') AS constructor_count
    FROM raw.constructor_standings
    ORDER BY season
""").df()


,season,constructor_count
0,2006,12
1,2007,11
2,2008,11
3,2009,10
4,2010,12
5,2011,12
6,2012,12
7,2013,11
8,2014,11
9,2015,10
